# Bump test -- rekon10 arm, struck by hand

`ringdown.py` applied to a capture where the arm was struck deliberately, to get the
frequencies and damping of whatever the structure rings at. The point is the prop decision:
if there is a structural resonance near the hover blade-pass or rev line, moving hover RPM
(8 vs 9 vs 10 inch, 2 vs 3 blade) either helps or lands on top of it, and guessing which is
not acceptable.

**What this capture is.** 260926, campod-sw on the bench, vehicle on battery so the FC logs.
The operator struck the end of the arm repeatedly with a metal hook while holding the vehicle
down. Two taps beforehand mark the start; six held poses follow, for the separate calibration
question.

**Two limits stated up front, because they bound what a mode frequency from this means.**

1. **The base was not clamped.** The vehicle was held down by hand, not fixtured, and it moved
   under the strikes. So a fitted mode is of the whole hand-restrained assembly, not of a
   cantilever with a fixed root. Frequencies from this are indicative; they are not the arm's
   clamped modes and should not be quoted as such.
2. **The strikes clip.** Peaks exceed the +/-16 g range that was configured. The decay is
   inside range, which is what the fit uses, but the first few milliseconds of each strike are
   not trustworthy and the strike amplitude is a lower bound.


In [ ]:
# papermill parameters
capture_dir = ("/home/jovyan/datasets/flights/260926-sixpose-and-bump/extracted/"
               "campod-sw/de59ef4b-37d4-41f2-b860-9532c76e233c")
flight_dir = "/home/jovyan/datasets/flights/260926-sixpose-and-bump"
out_dir = None
# The window the operator struck in. Set from the transient census below on a new capture;
# left explicit rather than auto-detected because the same record also contains taps, pose
# changes and handling, all of which are impulses and none of which are the bump test.
strike_window = (283.0, 303.0)

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib
# The inline backend, deliberately, and NOT Agg -- Agg renders into a buffer nobody collects,
# so the executed notebook comes out with no images in it.
%matplotlib inline
import matplotlib.pyplot as plt

REPO = Path.cwd()
while not (REPO / "analysis" / "ringdown.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "analysis"))
import ringdown as rd

CAP = Path(capture_dir)
FLIGHT = Path(flight_dir)
OUT = Path(out_dir) if out_dir else (FLIGHT / "derived")
OUT.mkdir(parents=True, exist_ok=True)
GRID = dict(color="#d9d8d2", lw=0.6, alpha=0.9)
COL = {"camera": "#2a78d6", "arm": "#eb6834"}

S = {}
for lab in ("camera", "arm"):
    t, x, y, z, h = rd.load_accel_jsonl(CAP / f"accel-{lab}.jsonl")
    S[lab] = dict(t=t, x=x, y=y, z=z, h=h, mag=np.sqrt(x*x + y*y + z*z))
    print(f"{lab:6s} {h['n_samples']:>9,d} samples  fitted {h['rate_fitted_hz']:8.2f} Hz  "
          f"boot {t[0]:6.1f}->{t[-1]:6.1f} s  range +/-{h['range_g']} g  "
          f"{h['scale_mg_per_lsb']} mg/LSB  truncated_lines={h['truncated_lines']}")
print(f"\nout dir: {OUT}")

## 1. Assumptions -- what has to hold before a frequency from this means anything

Checked here rather than assumed, because each of these has been wrong at least once.


In [ ]:
# Loss accounting. `n` < 32 is the NORMAL case -- the reader takes whatever the FIFO holds
# and the mean batch is 2-3 samples -- so it says nothing. `ovr` is only read when the FIFO
# came back full, so it is false by construction on a short batch and cannot classify either.
# What finds a hole is gap_ns against the FIFO fill time.
assume = {}
for lab in ("camera", "arm"):
    bt, bi, bn, gp, ov = [], [], [], [], []
    for line in open(CAP / f"accel-{lab}.jsonl"):
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        if (r.get("t") or r.get("type")) == "b":
            bt.append(r["boot_ns"] / 1e9); bi.append(r["i"]); bn.append(r["n"])
            gp.append(r["gap_ns"] / 1e9); ov.append(bool(r["ovr"]))
    bt = np.array(bt); bi = np.array(bi, float); bn = np.array(bn)
    gp = np.array(gp); ov = np.array(ov)
    rate = float(np.polyfit(bt, bi, 1)[0]); fill = 32 / rate
    big = gp > 2 * fill
    dark = float(np.clip(gp[big] - fill, 0, None).sum())
    span = float(bt[-1] - bt[0])
    S[lab].update(bt=bt, bi=bi, gap=gp, stall=big, fill=fill, t0=float(bt[0]))
    assume[lab] = dict(batches=int(len(bn)), mean_n=float(bn.mean()), fill_ms=fill * 1e3,
                       stalls=int(big.sum()), dark_s=dark, dark_frac=dark / span,
                       last_stall_s=float((bt[big] - bt[0]).max()) if big.any() else None)
    a = assume[lab]
    print(f"{lab:6s} {a['batches']:>7,d} batches, mean n {a['mean_n']:.2f}, FIFO fills in "
          f"{a['fill_ms']:.2f} ms")
    print(f"       {a['stalls']} stalls > 2x fill; {a['dark_s']:.2f} s of {span:.0f} s carries no "
          f"samples ({a['dark_frac']*100:.2f}%); last at +{a['last_stall_s']:.1f} s")

In [ ]:
# Is the strike window itself clean? A hole inside a decay would corrupt a fit silently.
W0, W1 = strike_window
for lab in ("camera", "arm"):
    b = S[lab]
    m = (b["bt"] - b["t0"] >= W0) & (b["bt"] - b["t0"] <= W1)
    worst = float(b["gap"][m].max())
    n_stall = int(b["stall"][m].sum())
    print(f"{lab:6s} strike window {W0:.0f}-{W1:.0f} s: {n_stall} stalls, "
          f"worst read gap {worst*1e3:.2f} ms against a {b['fill']*1e3:.2f} ms fill "
          f"({worst/b['fill']*100:.0f}% of it)")
    assume[lab]["window_stalls"] = n_stall
    assume[lab]["window_worst_gap_ms"] = worst * 1e3
    assert n_stall == 0, (f"{lab}: {n_stall} stalls inside the strike window -- a decay fitted "
                          "across a hole is not a decay")

In [ ]:
# The fitted rate is EMITTED SAMPLES PER SECOND OF Pi CLOCK_BOOTTIME. It is not the part's
# ODR and it is not referenced to anything absolute: any sample the reader failed to emit
# depresses it, and the only clock in it is the Pi's. So it is used for the spectrum's
# frequency axis and nothing is concluded from its drift.
print("30 s segment rates -- all segments, then only those with no stall:")
for lab in ("camera", "arm"):
    b = S[lab]; bt, bi = b["bt"], b["bi"]
    allr, clean = [], []
    for e in np.arange(bt[0], bt[-1] - 30, 30.0):
        m = (bt >= e) & (bt < e + 30)
        if m.sum() < 50:
            continue
        r = float(np.polyfit(bt[m], bi[m], 1)[0])
        allr.append(r)
        if not b["stall"][m].any():
            clean.append(r)
    f = lambda v: (f"{min(v):.1f}-{max(v):.1f} Hz, spread "
                   f"{(max(v)-min(v))/np.mean(v)*1e6:.0f} ppm" if len(v) > 1 else "n/a")
    print(f"  {lab:6s} all (n={len(allr)}): {f(allr)}")
    print(f"  {lab:6s} stall-free (n={len(clean)}): {f(clean)}")
    assume[lab]["rate_spread_all_ppm"] = (max(allr)-min(allr))/np.mean(allr)*1e6
    assume[lab]["rate_spread_clean_ppm"] = (max(clean)-min(clean))/np.mean(clean)*1e6
print("\nThe all-segment spread is dominated by stalls: a segment missing samples fits low,")
print("because the lost samples are absent from the count while their time still elapsed.")

In [ ]:
# Amplitude calibration, which the poses in this same capture exist to fix. Sitting still on
# a bench, |a| must read 1 g; the residual is scale-plus-bias error along whatever axis
# gravity happens to lie on, so this is a floor on the error, not the whole of it.
rest = (60.0, 220.0)   # inside the long upright hold, before the first tap
print(f"|a| over the upright hold {rest[0]:.0f}-{rest[1]:.0f} s (gravity only, must be 1.000 g):")
for lab in ("camera", "arm"):
    d = S[lab]; tt = d["t"] - d["t"][0]
    m = (tt >= rest[0]) & (tt <= rest[1])
    mu = float(d["mag"][m].mean()); sd = float(d["mag"][m].std())
    assume[lab]["rest_mag_g"] = mu
    assume[lab]["rest_mag_err_pct"] = (mu - 1.0) * 100
    print(f"  {lab:6s} {mu:.4f} g  ({(mu-1)*100:+.2f}%), sd {sd:.4f} g")
print("\nAny amplitude COMPARISON between the two sensors inherits the difference between")
print("those two errors until the six-pose fit lands. Frequencies do not -- a scale error")
print("multiplies the signal, it does not move a peak.")

In [ ]:
# Clipping. Peaks past the configured range are flat-topped, which puts energy at harmonics
# that are not in the structure. The rail is 4095 LSB, and load_accel_jsonl returns g, so the
# comparison has to be in g -- testing g against 4095 is a test that can never fire.
for lab in ("camera", "arm"):
    d = S[lab]; tt = d["t"] - d["t"][0]
    rail_g = 4095 * d["h"]["scale_mg_per_lsb"] / 1000.0
    rail = ((np.abs(d["x"]) >= rail_g) | (np.abs(d["y"]) >= rail_g) | (np.abs(d["z"]) >= rail_g))
    m = (tt >= W0) & (tt <= W1)
    assume[lab]["rail_g"] = rail_g
    assume[lab]["rail_frac_capture"] = float(rail.mean())
    assume[lab]["rail_frac_header"] = float(d["h"]["rail_hit_frac"])
    assume[lab]["rail_n_window"] = int(rail[m].sum())
    print(f"  {lab:6s} rail at {rail_g:.2f} g/axis; {rail.sum()} railed samples in the capture "
          f"({rail.mean()*100:.4f}%), {rail[m].sum()} of them inside the strike window")
    print(f"         loader's own rail_hit_frac (computed in LSB): "
          f"{d['h']['rail_hit_frac']*100:.4f}%  -- must agree with the line above")
    print(f"         peak |a| in the window {d['mag'][m].max():.1f} g, vector sum of three "
          f"axes each limited to {rail_g:.1f} g")
    assert abs(rail.mean() - d["h"]["rail_hit_frac"]) < 1e-6, (
        f"{lab}: rail fraction in g disagrees with the loader's LSB-domain figure")

## 2. Cross-sensor timing: can the two be compared at all?

The question is whether a frequency measured on one sensor is the same number as the same
frequency measured on the other. Two separate things go into that.

**Relative timing is structural, not fitted.** `load_accel_jsonl` places each sample backwards
from *its own batch's* `boot_ns` at the fitted rate, and a batch is 2-3 samples, so the time
axis is re-anchored to a kernel stamp roughly every 0.7 ms and rate error cannot accumulate --
a 2000 ppm rate error displaces a sample by 2000 ppm x 3 samples, which is 0.006 of a sample.
Both streams are stamped by the *same* kernel clock, so neither fitted rate enters their
relative alignment.

**So it is testable on a shared physical event.** A strike is broadband and arrives at both
sensors within about 25 us of each other (127 mm of aluminium at ~5000 m/s), which is well
inside one 0.3 ms sample. Comparing onsets across events that span the record therefore
measures relative clock drift down to a few ppm, and it does not depend on mode shape the way
a cross-correlation of the ringing itself would -- two sensors at different points on a
vibrating structure are legitimately out of phase, so a correlation lag there is physics, not
clock.

**Absolute frequency is a different matter and is not resolved here.** Everything is referred
to the Pi's `CLOCK_BOOTTIME`. There is no disciplined clock on this pod, and `wall_ns - boot_ns`
carries no independent information while NTP is not slewing. So a mode at 180 Hz means 180 Hz
*as this Pi counts seconds*.


In [ ]:
# Onsets of every impulse in the record, on each sensor, compared.
def onsets(lab, lo, hi, frac=0.15):
    """First crossing of `frac` of the window's own peak, per impulse group.

    Relative to each sensor's OWN peak, not an absolute level: the two sensors differ by 8.7%
    in scale and see different amplitudes at different points on the structure, so an absolute
    threshold crosses at genuinely different times on the same rising edge and the difference
    that comes out is the threshold's, not the clock's.
    """
    d = S[lab]; tt = d["t"] - d["t"][0]
    m = (tt >= lo) & (tt <= hi)
    a = d["mag"][m] - np.median(d["mag"][m])
    ts = tt[m]
    q = np.abs(a) > frac * np.abs(a).max()
    idx = np.flatnonzero(q)
    if not len(idx):
        return []
    out, run = [], [idx[0]]
    for i in idx[1:]:
        if ts[i] - ts[run[-1]] < 0.2:
            run.append(i)
        else:
            out.append(float(ts[run[0]])); run = [i]
    out.append(float(ts[run[0]]))
    return out

# Events spanning the record: the opening taps, the strikes, the handling before pose 1.
spans = [("taps", 222.0, 227.0), ("strikes", W0, W1), ("handling", 396.0, 399.0)]
rows = []
for name, lo, hi in spans:
    oc, oa = onsets("camera", lo, hi), onsets("arm", lo, hi)
    for a in oc:
        near = [b for b in oa if abs(b - a) < 0.05]
        if near:
            rows.append((name, a, min(near, key=lambda b: abs(b - a)) - a))
print(f"{len(rows)} impulses matched on both sensors across "
      f"{rows[-1][1] - rows[0][1]:.0f} s of record\n")
print("  group        t (s)     camera-minus-arm onset")
for name, a, dt in rows:
    print(f"  {name:10s} {a:8.2f}   {dt*1e3:+8.2f} ms")
dt = np.array([r[2] for r in rows]); te = np.array([r[1] for r in rows])
samp_ms = 1e3 / S["camera"]["h"]["rate_fitted_hz"]
print(f"\nmedian {np.median(dt)*1e3:+.2f} ms, spread {dt.std()*1e3:.2f} ms, "
      f"one sample = {samp_ms:.2f} ms")
if len(rows) > 2:
    slope = float(np.polyfit(te, dt, 1)[0])
    print(f"drift of that offset across the record: {slope*1e6:+.3f} us/s = {slope*1e6:+.2f} ppm")
    print("Relative drift at this level does not move a mode frequency measurably; the two")
    print("sensors are on one timebase and can be compared directly.")
assume["cross_sensor"] = dict(n_matched=len(rows), median_offset_ms=float(np.median(dt)*1e3),
                              spread_ms=float(dt.std()*1e3), sample_ms=samp_ms,
                              drift_ppm=float(slope*1e6) if len(rows) > 2 else None)

## 3. The record contains the procedure that was logged

Shown, not assumed: the held poses and the transient census should match the sequence in the
flight README. If they do not, the window every fit below runs in is wrong.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, lab in zip(axes, ("camera", "arm")):
    d = S[lab]; tt = d["t"] - d["t"][0]
    rate = d["h"]["rate_fitted_hz"]; w = max(1, int(0.05 * rate))
    k = len(d["mag"]) // w
    env = np.abs(d["mag"][:k*w] - np.median(d["mag"])).reshape(k, w).max(axis=1)
    ax.plot(tt[:k*w:w], env, lw=0.5, color=COL[lab])
    ax.axvspan(W0, W1, color="#eb6834", alpha=0.12, label="strike window")
    ax.set_yscale("log"); ax.set_ylabel(f"{lab}\n|a| - median (g)")
    ax.grid(**GRID); ax.legend(loc="upper left", fontsize=8)
axes[0].set_title("260926 campod-sw: 50 ms peak envelope over the whole capture")
axes[-1].set_xlabel("seconds from capture start")
fig.tight_layout(); fig.savefig(OUT / "bump-capture-overview.png", dpi=130); plt.show()

# Held plateaus, as the poses for the separate calibration question.
print("held plateaus (1 s window sd < 0.05 g, runs >= 8 s), camera:")
d = S["camera"]; rate = int(d["h"]["rate_fitted_hz"]); k = len(d["mag"]) // rate
m1 = d["mag"][:k*rate].reshape(k, rate)
quiet = m1.std(axis=1) < 0.05
runs, s = [], None
for i, q in enumerate(quiet):
    if q and s is None:
        s = i
    elif not q and s is not None:
        if i - s >= 8:
            runs.append((s, i))
        s = None
if s is not None and k - s >= 8:
    runs.append((s, k))
ax_mean = [S["camera"][c][:k*rate].reshape(k, rate).mean(axis=1) for c in ("x", "y", "z")]
for a, b in runs:
    print(f"  {a:4d}-{b:4d} s ({b-a:3d} s)  |a|={m1[a:b].mean():.3f} g   "
          f"x={ax_mean[0][a:b].mean():+.3f} y={ax_mean[1][a:b].mean():+.3f} "
          f"z={ax_mean[2][a:b].mean():+.3f}")
print(f"\n{len(runs)} plateaus. The long first one is the upright hold; the short ones after")
print("the strikes are the five remaining poses.")

## 4. Ringdown

`ringdown.analyze` finds the impulses, picks candidate bands from the record's own spectrum,
and fits an exponential decay per band per impulse. `summarize` then groups fits into modes
and reports the scatter ACROSS impulses -- which is the thing that says whether a number is a
property of the structure or of one swing of a hook.


In [ ]:
W = {}
for lab in ("camera", "arm"):
    d = S[lab]; tt = d["t"] - d["t"][0]
    m = (tt >= W0) & (tt <= W1)
    W[lab] = dict(t=d["t"][m], mag=d["mag"][m] - np.median(d["mag"][m]),
                  rate=d["h"]["rate_fitted_hz"])
    chk = rd.looks_like_a_bump_test(W[lab]["mag"])
    print(f"{lab:6s} impulses-in-silence check: {chk}")
    assert chk["ok"], f"{lab}: window is not impulses-in-silence"

### Choosing the impulse threshold, against an independent count

`find_taps` groups samples that sit above a MAD-derived floor, splitting on quiet gaps. At the
default `thresh_sd=6` the camera returns 2 impulses where the arm returns 5, for a window that
contains the same physical strikes: the camera's decay stays above 6x its floor between blows,
so consecutive strikes merge into one group and only the loudest of each merged run gets fitted.
That is a threshold artefact and it would silently halve the statistics on one sensor.

So the threshold is chosen rather than defaulted, against a count made a different way -- a 50 ms
peak envelope over the window, grouped at 0.3 s -- and the notebook asserts both sensors find
the same number. If they disagree, no cross-sensor comparison below is meaningful.


In [ ]:
def census(lab):
    """Independent impulse count: 50 ms peak envelope, grouped at 0.3 s."""
    w = W[lab]; rate = w["rate"]
    b = max(1, int(0.02 * rate)); k = len(w["mag"]) // b
    pk = np.abs(w["mag"][:k*b]).reshape(k, b).max(axis=1)
    tt = w["t"][:k*b:b] - w["t"][0]
    hot = np.flatnonzero(pk > 0.25 * pk.max())
    if not len(hot):
        return []
    out, run = [], [hot[0]]
    for i in hot[1:]:
        if tt[i] - tt[run[-1]] < 0.3:
            run.append(i)
        else:
            out.append(float(tt[run[0]])); run = [i]
    out.append(float(tt[run[0]]))
    return out

N_CENSUS = {lab: len(census(lab)) for lab in ("camera", "arm")}
print(f"envelope census: camera {N_CENSUS['camera']} impulses, arm {N_CENSUS['arm']}")

print("\nfind_taps count vs thresh_sd:")
THRESH = None
for ts in (6, 10, 20, 40, 60, 80, 120):
    n = {lab: len(rd.find_taps(W[lab]["t"], W[lab]["mag"], thresh_sd=ts))
         for lab in ("camera", "arm")}
    mark = ""
    agree = n["camera"] == n["arm"]
    near = abs(n["camera"] - N_CENSUS["camera"]) <= 1
    if THRESH is None and agree and near:
        THRESH = ts; N_FOUND = n["camera"]; mark = "   <- chosen"
    print(f"  thresh_sd={ts:4d}  camera {n['camera']:3d}  arm {n['arm']:3d}{mark}")
# What has to hold is that the two sensors fit the SAME impulses, so a mode found on one can be
# compared with the same mode on the other. Exact agreement with the envelope census is NOT
# required and would be the wrong bar: they are different detectors with different grouping, so
# one differing by a single weak strike is them disagreeing about a threshold, not about the
# record. Within one is the bar; the two sensors matching each other is the requirement.
assert THRESH is not None, (
    f"no thresh_sd makes the two sensors find the same number of impulses within 1 of the "
    f"census {N_CENSUS} -- the impulse sets are not comparable, so neither are the fits")
print(f"\nusing thresh_sd={THRESH}: {N_FOUND} impulses on both sensors, against an envelope "
      f"census of {N_CENSUS['camera']}")
print("The count is not monotonic in the threshold -- at 6 the camera's decay never drops below")
print("the floor so strikes merge into 2 groups, at 10 the floor lands mid-decay and one strike")
print("splits into several. That non-monotonicity is why the threshold is picked against an")
print("independent count rather than tuned until the answer looks right.")

### Two band searches, because one of them cannot see the arm

`candidate_bands` ranks peaks by prominence over the median and keeps the strongest. With the
base hand-held that ranking is decided entirely by the low-frequency response of a vehicle
moving in someone's grip: every one of the six bands it returns on both sensors falls below
100 Hz. The arm's own bending range never gets a band, not because nothing is there but
because it is not in the top six.

So the search is run twice and the results are kept apart: once as the tool ranks it, and once
restricted to 120 Hz and up. The second is not a better answer -- it is the only one that
addresses the prop question, since hover blade-pass is in the hundreds of Hz.


In [ ]:
SEARCHES = {"peak_ranked": dict(fmin=30.0), "structural": dict(fmin=120.0)}
results, summaries = {}, {}
for lab in ("camera", "arm"):
    w = W[lab]
    results[lab], summaries[lab] = {}, {}
    for name, kw in SEARCHES.items():
        bands = rd.candidate_bands(w["t"], w["mag"], fmax=w["rate"] / 2.5, n=6, **kw)
        print(f"\n=== {lab} / {name}: bands from the window spectrum ===")
        for b in bands:
            print(f"   {b[0]:8.1f} - {b[1]:8.1f} Hz")
        if not bands:
            results[lab][name], summaries[lab][name] = {}, {}
            print("   (no band cleared the prominence floor in this range)")
            continue
        res = rd.analyze(w["t"], w["mag"], bands=bands, tap_kw=dict(thresh_sd=THRESH))
        results[lab][name] = res
        summaries[lab][name] = rd.summarize(res)
        n_modes = len(summaries[lab][name].get("modes", []))
        print(f"   {len(res.get('taps', []))} impulses fitted, {n_modes} modes survive")
        for mode in summaries[lab][name].get("modes", []):
            print(f"      {mode.get('f_hz', float('nan')):8.2f} Hz  "
                  f"+/-{mode.get('f_sd_hz', float('nan')):.2f}  "
                  f"zeta {mode.get('zeta', float('nan')):.4f}  "
                  f"Q {mode.get('q', float('nan')):6.1f}  "
                  f"decay {mode.get('decay_s', float('nan'))*1e3:6.1f} ms  "
                  f"r2 {mode.get('r2_median', float('nan')):.3f}  "
                  f"n_taps {mode.get('n_taps')}")

In [ ]:
# Spectrum of the strike window, both sensors, with the surviving modes marked. This is the
# picture the numbers above come from, so it is here to be disagreed with.
from scipy.signal import welch
fig, ax = plt.subplots(figsize=(14, 5))
for lab in ("camera", "arm"):
    w = W[lab]
    f, P = welch(w["mag"], fs=w["rate"], nperseg=4096)
    ax.semilogy(f, P, lw=0.8, color=COL[lab], label=f"{lab} (fitted {w['rate']:.0f} Hz)")
    for name in SEARCHES:
        for mode in summaries[lab].get(name, {}).get("modes", []):
            fq = mode.get("f_hz") or mode.get("f_mean_hz")
            if not fq:
                continue
            ax.axvline(fq, color=COL[lab], ls="--" if name == "structural" else ":",
                       lw=0.9, alpha=0.7)
            ax.annotate(f"{fq:.0f}", (fq, ax.get_ylim()[1]), fontsize=7,
                        color=COL[lab], rotation=90, va="top")
ax.set_xlim(0, 1400); ax.set_xlabel("Hz"); ax.set_ylabel("PSD of |a| (g^2/Hz)")
ax.set_title(f"Strike window {W0:.0f}-{W1:.0f} s: spectrum and fitted modes "
             "(dashed). Base hand-held, not clamped.")
ax.grid(**GRID); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "bump-spectrum-and-modes.png", dpi=130); plt.show()

In [ ]:
# One strike, close up, on both sensors: the raw decay the fit is fitting.
i0 = 285.7
fig, ax = plt.subplots(figsize=(14, 4))
for lab in ("camera", "arm"):
    d = S[lab]; tt = d["t"] - d["t"][0]
    m = (tt >= i0 - 0.005) & (tt <= i0 + 0.20)
    ax.plot((tt[m] - i0) * 1e3, d["mag"][m] - np.median(d["mag"]), lw=0.7,
            color=COL[lab], label=lab)
ax.axhline(0, color="#999", lw=0.5)
ax.set_xlabel("ms from strike"); ax.set_ylabel("|a| - median (g)")
ax.set_title(f"Strike at {i0:.1f} s. The first few ms are clipped at +/-16 g per axis; "
             "the fit uses the decay after it.")
ax.grid(**GRID); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "bump-single-strike.png", dpi=130); plt.show()

## 5. Do the two sensors see the same structure?

They sit on the same arm about 127 mm apart, so they should report the same mode frequencies
and different amplitudes -- a mode has one frequency and a shape. Frequencies agreeing is the
check that the modes are real; amplitudes differing is the measurement.


In [ ]:
try:
    paired = {name: rd.compare_arms(summaries["camera"].get(name, {}),
                                    summaries["arm"].get(name, {}))
              for name in SEARCHES}
    print(json.dumps(paired, indent=1, default=float))
except Exception as exc:
    paired = {"error": str(exc)}
    print(f"compare_arms: {exc}")
    print("(compare_arms pairs two nominally identical arms; these two sensors are on the SAME")
    print(" arm, so a failure here is about the function's intent, not about the data.)")

## 6. Where these modes sit relative to what actually drives the vehicle

This is the reason the bump test exists. A mode only matters if something excites it, and what
excites this airframe is the rotors: each motor puts a line at its rev frequency and at
multiples of it, blade count included. So the question is arithmetic -- how far would hover RPM
have to move for a forcing line to land on a measured mode.

The forcing lines are taken from the 260923 flight's own measured spectrum, not from nominal
figures. Prop changes move hover RPM: a smaller prop has to spin faster for the same thrust, so
going down in diameter walks every line up, and going to fewer blades walks the blade-pass line
down relative to the rev line.

**What this cannot tell you.** The base was hand-held, so 153 Hz is the mode of the
hand-restrained assembly. On a flying vehicle the boundary condition at the arm root is
different and the frequency will not be identical. The distance to a forcing line is what is
being computed, and the mode frequency going into it carries that uncertainty.


In [ ]:
prev = Path("/home/jovyan/datasets/flights/rekon10/260923-new-props/derived/"
            "vibration-spectrogram.json")
lines = []
if prev.exists():
    pj = json.loads(prev.read_text())
    for row in pj.get("line_amplitudes", []) or pj.get("stationary_window", {}).get("line_table", []):
        hz = row.get("hz") or row.get("pred_hz")
        if hz:
            lines.append((hz, row.get("motor") or row.get("label") or "", row.get("order")))
    lines = sorted({round(h, 2): (h, m, o) for h, m, o in lines}.values())
    print(f"forcing lines measured in hover on 260923 ({prev.name}):")
    for h, m, o in lines:
        print(f"   {h:8.2f} Hz   {m}  order={o}")
else:
    print(f"no 260923 derived spectrum at {prev} -- forcing comparison skipped")

# Modes worth comparing: only those seen on more than one strike, and preferably on both
# sensors. A single-tap fit is one swing of a hook, not a property of the structure.
solid = []
for lab in ("camera", "arm"):
    for name in SEARCHES:
        for mode in summaries[lab].get(name, {}).get("modes", []):
            if (mode.get("n_taps") or 0) >= 3:
                solid.append(dict(sensor=lab, search=name, **mode))
print(f"\nmodes fitted on 3 or more strikes: {len(solid)}")
for m in solid:
    print(f"   {m['sensor']:6s} {m['search']:11s} {m['f_hz']:8.2f} Hz  "
          f"+/-{m.get('f_sd_hz', float('nan')):.2f}  Q {m.get('q', float('nan')):5.1f}  "
          f"n_taps {m['n_taps']}  r2 {m.get('r2_median', float('nan')):.3f}")

# Modes both sensors independently agree on are the defensible ones.
agreed = []
for a in [m for m in solid if m["sensor"] == "camera"]:
    for b in [m for m in solid if m["sensor"] == "arm"]:
        if abs(a["f_hz"] - b["f_hz"]) / a["f_hz"] < 0.01:
            agreed.append((a, b))
print(f"\nmodes BOTH sensors found independently (within 1%): {len(agreed)}")
for a, b in agreed:
    d_hz = abs(a["f_hz"] - b["f_hz"])
    print(f"   {a['f_hz']:.2f} Hz (camera) vs {b['f_hz']:.2f} Hz (arm): "
          f"{d_hz:.3f} Hz apart = {d_hz/a['f_hz']*1e6:.0f} ppm")
    print(f"      damping differs: camera zeta {a['zeta']:.4f} (Q {a['q']:.1f}, decay "
          f"{a['decay_s']*1e3:.1f} ms) vs arm zeta {b['zeta']:.4f} (Q {b['q']:.1f}, decay "
          f"{b['decay_s']*1e3:.1f} ms)")
    print(f"      Same frequency from two sensors on one arm is the same mode. The damping")
    print(f"      difference is a property of each SENSOR's mount, not of the structure --")
    print(f"      the camera sensor is taped on and the tape is a lossy path.")

In [ ]:
# How far hover RPM would have to move for each forcing line to land on each solid mode.
if lines and solid:
    modes_u = sorted({round(m["f_hz"], 1) for m in solid})
    print("RPM change that would put a measured forcing line onto a measured mode")
    print("(positive = rotors must speed up, which is the direction a SMALLER prop goes)\n")
    print(f"  {'mode':>9s}  {'forcing line':>13s}  {'shift needed':>13s}   {'note'}")
    for fm in modes_u:
        for h, m, o in lines:
            shift = (fm / h - 1) * 100
            if abs(shift) > 60:
                continue          # beyond any prop change on this airframe
            flag = "  <-- inside a plausible prop change" if abs(shift) <= 30 else ""
            print(f"  {fm:7.1f} Hz  {h:10.2f} Hz  {shift:+11.1f} %   {m}{flag}")
    print("\nA line already sitting close to a mode means the CURRENT configuration is near it;")
    print("a small shift means a prop change could walk onto it. Neither is a recommendation --")
    print("which way to move, and whether to move at all, is a decision this notebook does not")
    print("make. The numbers are here so it is made against measurements.")
else:
    print("nothing to compare")

## 6. Agent-readable output


In [ ]:
result = dict(
    capture=str(CAP),
    session=S["camera"]["h"]["session"],
    node=S["camera"]["h"]["node"],
    started_utc=S["camera"]["h"]["started_utc"],
    strike_window_s=[W0, W1],
    # Stated with the numbers so a consumer cannot take a mode frequency for a clamped one.
    caveats=dict(
        base_clamped=False,
        base_note=("vehicle held down by hand and moving under the strikes; modes are of the "
                   "hand-restrained assembly, not of a cantilever with a fixed root"),
        peaks_clipped=True,
        absolute_frequency_reference="Pi CLOCK_BOOTTIME; no disciplined clock on this pod",
        band_search_note=("peak-ranked bands all fall below 100 Hz because the hand-held base "
                          "dominates the spectrum; the structural search is the one that "
                          "addresses blade-pass"),
    ),
    assumptions=assume,
    per_sensor={lab: dict(
        rate_fitted_hz=S[lab]["h"]["rate_fitted_hz"],
        n_samples=S[lab]["h"]["n_samples"],
        modes_by_search={name: summaries[lab].get(name, {}).get("modes", [])
                         for name in SEARCHES},
        n_impulses={name: len(results[lab].get(name, {}).get("taps", []))
                    for name in SEARCHES},
        impulse_census=N_CENSUS[lab],
        thresh_sd_used=THRESH,
    ) for lab in ("camera", "arm")},
    cross_sensor_modes=paired,
    modes_on_3_or_more_strikes=solid,
    modes_agreed_by_both_sensors=[dict(camera=a, arm=b) for a, b in agreed],
    forcing_lines_260923_hz=[h for h, _, _ in lines],
)
p = OUT / "bump-test-ringdown.json"
p.write_text(json.dumps(result, indent=1, default=float))
print(f"wrote {p}")
print(json.dumps(result, indent=1, default=float)[:2500])